In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [4]:
from anthropic.types import ToolParam

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

In [5]:
def run_get_current_datetime_tool():
    messages = []
    messages.append({
        "role": "user",
        "content": "what is the exact time, formatted as HH:MM:SS?"
    })

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=[get_current_datetime_schema]
    )

    messages.append({
        "role": "assistant",
        "content": response.content
    })

    tool_use_block = next(
        (block for block in response.content if getattr(block, "type", None) == "tool_use"),
        None
    )

    if tool_use_block is None:
        raise RuntimeError("No tool_use block returned from the first call")

    result = get_current_datetime(**tool_use_block.input)

    messages.append({
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_use_block.id,
                "content": str(result),
                "is_error": False
            }
        ]
    })

    followup = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=[get_current_datetime_schema]
    )

    print(followup.content)

run_get_current_datetime_tool()


[TextBlock(citations=None, text='The exact time is **21:25:27** (9:25:27 PM in 12-hour format).', type='text')]


In [6]:
# Default format: "2024-01-15 14:30:25"
get_current_datetime()

# Just hour and minute: "14:30"
get_current_datetime("%H:%M")

'21:25'

In [8]:
# examples: weather location details

def get_weather(location):
   if not location or location.strip() == "":
       raise ValueError("Location cannot be empty or None")
   
   url = "https://fakeweatherapi.example.com/current"
   params = {
       "q": location,
       "appid": api_key,
       "Units": "metric"  # or "imperial" for Fahrenheit
   }

   response = requests.get(url, params=params, timeout=10)
   response.raise_for_status()  # Raise an error for bad responses

   return response.json()  # Assuming the API returns JSON data


In [1]:
from anthropic import Anthropic

client = Anthropic()

# Pretend "database" standing in for your 5 course contracts
CONTRACTS = {
    "nimbus cloud services": "Auto-renews Sept 25, 2026 unless cancelled by Aug 26.",
    "brightpath security solutions": "Auto-renews Oct 10, 2026 unless cancelled by Aug 26.",
}

def check_contract_status(vendor_name: str) -> str:
    return CONTRACTS.get(vendor_name.lower(), "No record found for that vendor.")

tools = [{
    "name": "check_contract_status",
    "description": "Look up the renewal status of a vendor contract by vendor name.",
    "input_schema": {
        "type": "object",
        "properties": {"vendor_name": {"type": "string"}},
        "required": ["vendor_name"],
    },
}]

messages = [{"role": "user", "content": "What's the renewal status of the Nimbus Cloud Services contract?"}]

response = client.messages.create(
    model="claude-sonnet-5", max_tokens=500, tools=tools, messages=messages,
)
messages.append({"role": "assistant", "content": response.content})

if response.stop_reason == "tool_use":
    tool_block = next(b for b in response.content if b.type == "tool_use")
    result = check_contract_status(**tool_block.input)
    messages.append({
        "role": "user",
        "content": [{"type": "tool_result", "tool_use_id": tool_block.id, "content": result}],
    })
    final = client.messages.create(model="claude-sonnet-5", max_tokens=500, tools=tools, messages=messages)
    print(next(b.text for b in final.content if b.type == "text"))

Here's the renewal status for **Nimbus Cloud Services**:

- **Status:** Set to auto-renew
- **Renewal Date:** September 25, 2026
- **Cancellation Deadline:** You must cancel by **August 26, 2026** if you don't want it to renew automatically.

Let me know if you'd like help tracking this deadline or reviewing the contract terms further.


In [2]:
tools.append({
    "name": "get_todays_date",
    "description": "Returns today's date.",
    "input_schema": {"type": "object", "properties": {}},
})

messages = [{"role": "user", "content": (
    "What's today's date, and what's the renewal status of the "
    "Nimbus Cloud Services contract?"
)}]
response = client.messages.create(model="claude-sonnet-5", max_tokens=500, tools=tools, messages=messages)
tool_uses = [b for b in response.content if b.type == "tool_use"]
print(f"Claude requested {len(tool_uses)} tool call(s) in this turn: {[t.name for t in tool_uses]}")

Claude requested 2 tool call(s) in this turn: ['get_todays_date', 'check_contract_status']


In [4]:
import base64
from anthropic import Anthropic

client = Anthropic()

with open("contract_screenshot.png", "rb") as f:
    image_data = base64.b64encode(f.read()).decode("utf-8")

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=500,
    messages=[{
        "role": "user",
        "content": [
            {"type": "image", "source": {"type": "base64", "media_type": "image/png", "data": image_data}},
            {"type": "text", "text": "What vendor and end date does this contract show?"},
        ],
    }],
)
print(next(b.text for b in response.content if b.type == "text"))

According to the Service Agreement, the vendor and end date are:

- **Vendor:** Meridian Cloud Solutions Inc.
- **End Date:** December 31, 2027


In [13]:
from anthropic import Anthropic

client = Anthropic()

file_obj = client.beta.files.upload(
    file=open("nimbus_contract.pdf", "rb"),
)
print(f"Uploaded. file_id = {file_obj.id}")

response = client.beta.messages.create(
    model="claude-sonnet-5",
    max_tokens=500,
    betas=["files-api-2025-04-14"],
    messages=[{
        "role": "user",
        "content": [
            {"type": "document", "source": {"type": "file", "file_id": file_obj.id}},
            {"type": "text", "text": "Summarize this contract in two sentences."},
        ],
    }],
)
print(next(b.text for b in response.content if b.type == "text"))

Uploaded. file_id = file_011Ce3pRhxQQW1qvbP7DM8nU
This is a fictional sample service agreement (Contract Reference: NCS-2025-0142) between an organization and Nimbus Cloud Services for primary cloud infrastructure hosting—covering compute, storage, and managed database services for production workloads—running for a 12-month term from September 25, 2025 to September 25, 2026 at an annual value of $63,000. The contract automatically renews for successive 12-month terms unless either party gives written cancellation notice at least 30 days before the end date.

Note: This document is explicitly labeled as a fictional sample created for AI training purposes and does not represent a real vendor, contract, or company.


In [17]:
from anthropic import Anthropic

client = Anthropic()

file_obj = client.beta.files.upload(file=open("contracts.csv", "rb"))

response = client.beta.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=2048,
    betas=[
        "files-api-2025-04-14",
        "code-execution-2025-05-22",
        "context-management-2025-06-27",  # required beta header for context editing
    ],
    tools=[{"type": "code_execution_20250522", "name": "code_execution"}],
    context_management={
        "edits": [
            {
                "type": "clear_tool_uses_20250919",
                "trigger": {"type": "input_tokens", "value": 3000},
            }
        ]
    },
    messages=[{
        "role": "user",
        "content": [
            {"type": "container_upload", "file_id": file_obj.id},
            {"type": "text", "text": (
                "Using this CSV, write and run Python to calculate total "
                "annual_value for every contract where auto_renews is TRUE "
                "and notice_days requires cancellation before Sept 1, 2026 "
                "(assume today is August 1, 2026). Show your code and the result."
            )},
        ],
    }],
)

for block in response.content:
    if block.type == "text":
        print(block.text)

# Inspect whether/how context clearing was applied
if hasattr(response, "context_management") and response.context_management:
    print("\n--- context_management.applied_edits ---")
    print(response.context_management.applied_edits)

## Results

**Total Annual Value of Qualifying Contracts: $87,000.00**

---

### How the Filter Works

The logic has two conditions that must **both** be true:

| Condition | Rule |
|---|---|
| `auto_renews == TRUE` | Contract automatically renews |
| `cancellation_deadline < Sept 1, 2026` | Must send cancellation notice *before* Sept 1, 2026 |

The **cancellation deadline** is calculated as:
> `cancellation_deadline = end_date − notice_days`

This tells us the *last possible day* to submit a cancellation notice before the contract rolls over.

---

### Qualifying Contracts

| Vendor | End Date | Notice Days | Cancellation Deadline | Annual Value |
|---|---|---|---|---|
| Nimbus Cloud Services | 2026-09-25 | 30 | **2026-08-26** ✅ | $63,000 |
| BrightPath Security Solutions | 2026-10-10 | 45 | **2026-08-26** ✅ | $24,000 |
| **Total** | | | | **$87,000** |

---

### Why the Others Were Excluded

| Vendor | Reason |
|---|---|
| Vertex Networking Group | `auto_renews=TRUE` but cancellation